# PCA + SVM training pipeline

This notebook downloads the diabetes dataset from Kaggle, applies PCA to keep 90% variance, and runs GridSearchCV to find the best SVM hyperparameters.


In [ ]:
from pathlib import Path
import pandas as pd
import kagglehub

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC


In [ ]:
path = kagglehub.dataset_download("neurocipher/breast-cancer-dataset")
print("Path to dataset files:", path)

csv_files = list(Path(path).rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found in the dataset directory.")

data_path = None
for csv_path in csv_files:
    if any(key in csv_path.name.lower() for key in ["breast", "cancer"]):
        data_path = csv_path
        break
if data_path is None:
    data_path = csv_files[0]

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print(df.head())
print(df.info())


In [ ]:
target_col = "diagnosis"
drop_cols = [target_col]
if "id" in df.columns:
    drop_cols.append("id")
X = df.drop(columns=drop_cols)
y = df[target_col]

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Target column:", target_col)
print("Train size:", X_train.shape, "Test size:", X_test.shape)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

transformers = []
if numeric_cols:
    transformers.append(("num", numeric_transformer, numeric_cols))
if categorical_cols:
    transformers.append(("cat", categorical_transformer, categorical_cols))
if not transformers:
    raise ValueError("No features found to train on.")

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("scale", StandardScaler(with_mean=True)),
    ("pca", PCA(n_components=0.90, svd_solver="full")),
    ("svc", SVC())
])

param_grid = {
    "svc__kernel": ["rbf", "linear"],
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", "auto"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)


In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification report:\n", classification_report(y_test, y_pred))
print("PCA components:", best_model.named_steps["pca"].n_components_)
print("Explained variance ratio sum:", best_model.named_steps["pca"].explained_variance_ratio_.sum())


In [ ]:
import joblib

output_dir = Path("artifacts")
output_dir.mkdir(exist_ok=True)

joblib.dump(best_model, output_dir / "pca_svm_pipeline.joblib")
joblib.dump(best_model.named_steps["pca"], output_dir / "pca.joblib")
joblib.dump(best_model.named_steps["svc"], output_dir / "svm.joblib")
print("Saved models to:", output_dir.resolve())
